In [25]:
import sys
from pathlib import Path
import torch

ROOT = Path.cwd().parent  # notebook/ 的 parent
sys.path.insert(0, str(ROOT))

import numpy as np

from dlphys.analysis.dmft.solver import DMFTConfig, DMFTSolver
from dlphys.analysis.dmft.softmax_moments import estimate_M_from_S
from dlphys.analysis.dmft.observables import residual_memory, corr_length_exp, power_law_alpha
from dlphys.analysis.dmft.stability import finite_diff_jacobian, spectral_radius

# -------------------------
# Benchmark A/B: M(C) sanity
# -------------------------
L = 8
cfg = DMFTConfig(L=L, gamma=0.5, n_mc=50000, mc_seed=0, damping=1)
solver = DMFTSolver(cfg)

# pick a simple C: exponential-ish
C = np.exp(-np.arange(L+1)/3.0)
S = solver.compute_S(C)

M = estimate_M_from_S(S, n_mc=80000, seed=0)
print("M symmetry error:", np.max(np.abs(M - M.T)))
print("sum M:", M.sum())

# scaling-degenerate: shrink logits variance -> uniform softmax
S_small = 1e-6 * S
M_small = estimate_M_from_S(S_small, n_mc=120000, seed=0)
M_target = np.ones((L+1, L+1)) / (L+1)**2
print("uniform M error:", np.max(np.abs(M_small - M_target)))
print("sum M_small:", M_small.sum())

# -------------------------
# Benchmark C: fixed point iteration
# -------------------------
def run_once(C0):
    out = solver.iterate(C0, verbose=False)
    Cstar = out["C_star"]
    return out["converged"], Cstar, out["err_hist"]

C0a = 0.2 * np.ones(L+1)
C0b = np.exp(-np.arange(L+1)/2.0)

conv_a, Cstar_a, err_a = run_once(C0a)
conv_b, Cstar_b, err_b = run_once(C0b)

print("converged A/B:", conv_a, conv_b)
print("Cstar distance:", np.max(np.abs(Cstar_a - Cstar_b)))

print("q:", residual_memory(Cstar_a))
print("xi(exp):", corr_length_exp(Cstar_a))
print("alpha(power):", power_law_alpha(Cstar_a))

M symmetry error: 0.0
sum M: 0.9999999999999999
uniform M error: 5.4467231653740344e-08
sum M_small: 1.0
converged A/B: True True
Cstar distance: 3.09172704343323e-08
q: 2.5690929413809313e-07
xi(exp): 8.795853185513511
alpha(power): 0.35419340834227686


In [31]:
# ============================================
# 0) Notebook environment setup
# ============================================
import sys
from pathlib import Path

ROOT = Path.cwd().parent  # notebooks/ 的 parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from dlphys.analysis.dmft.solver import DMFTConfig, DMFTSolver
from dlphys.analysis.dmft.init_C import INIT_REGISTRY   # 各种 C 初始化函数
from dlphys.analysis.dmft.attention_observables import focus_sharpness, sanity_sum_rule
from dlphys.analysis.dmft.classify_phase import classify_phase


ModuleNotFoundError: No module named 'dlphys.analysis.dmft.attention_observables'

In [ ]:

# 如果你已有这些（你之前的代码里导入过），就用；没有的话我下面也给了替代实现
try:
    from dlphys.analysis.dmft.observables import residual_memory, corr_length_exp, power_law_alpha
    HAVE_OLD_OBS = True
except Exception:
    HAVE_OLD_OBS = False


def residual_memory_fallback(C: np.ndarray) -> float:
    """有限L下最简单：取尾部一个点/或均值。这里用最后一个点。"""
    return float(C[-1])

# 只做最基本的指数/幂律拟合（作为 fallback）
def fit_exp_xi(C: np.ndarray, q: float = 0.0, lmin: int = 1) -> float | None:
    y = C - q
    L = len(C) - 1
    l = np.arange(L+1)
    mask = (l >= lmin) & (y > 1e-12)
    if mask.sum() < 3:
        return None
    x = l[mask].astype(float)
    z = np.log(y[mask])
    a, b = np.polyfit(x, z, 1)  # z ~ a l + b
    if a >= -1e-12:
        return None
    return float(-1.0/a)

def fit_power_alpha(C: np.ndarray, q: float = 0.0, lmin: int = 1) -> float | None:
    y = C - q
    L = len(C) - 1
    l = np.arange(L+1)
    mask = (l >= max(lmin, 1)) & (y > 1e-12)
    if mask.sum() < 3:
        return None
    x = np.log(l[mask].astype(float))
    z = np.log(y[mask])
    a, b = np.polyfit(x, z, 1)  # z ~ a log l + b
    return float(-a)


# ============================================
# 1) Model config + Initial condition choice
# ============================================

gamma = 0.5     # 固定 gamma
Ls = list(range(4, 65, 4))   # 扫描 L 的列表（你可以改范围/步长）

# Monte Carlo settings (M(C) 估计用)
n_mc = 60000          # 你可以加大来更平滑，但会更慢
mc_seed = 0
mc_batch = 2000

# DMFT iteration settings
damping = 0.3
tol = 1e-7
max_iter = 400

# ---- 选择 C 初始化 ansatz ----
# 你可以改 init_name: "exp", "plateau_exp", "damped_cos", "power_law", "gaussian", "spectrum"
init_name = "exp"
init_fn = INIT_REGISTRY[init_name]

# ---- 选择该 ansatz 的参数（你可以随意改）----
# 不同 ansatz 的参数不同，请看 init_C.py 的函数签名
# 例：exp: xi; plateau_exp: q, xi; spectrum: seed, smooth
init_kwargs = dict(xi=3.0, C0=1.0)

# 如果你想测试“多稳态”，最有效做法是：
# 1) 固定 (L,gamma)；2) 用多组 init_kwargs 或多种 init_name 重复跑；
# 3) 看是否收敛到不同 C*


# ============================================
# 2) DMFT iteration runner
# ============================================

def run_dmft_for_L(L: int) -> dict:
    """
    For given L:
      - initialize C0 by chosen ansatz
      - iterate DMFT to get C*
      - compute S(C*), M(C*), and some observables
    """
    cfg = DMFTConfig(
        L=L,
        gamma=gamma,
        n_mc=n_mc,
        mc_seed=mc_seed,
        damping=damping,
        tol=tol,
        max_iter=max_iter,
    )
    # 如果你的 DMFTConfig 支持 mc_batch，就加上（你之前 softmax_moments 有 batch 参数）
    if hasattr(cfg, "mc_batch"):
        cfg.mc_batch = mc_batch

    solver = DMFTSolver(cfg)

    # --- init C0 ---
    C0 = init_fn(L=L, **init_kwargs)

    # --- iterate ---
    out = solver.iterate(C0, verbose=False)
    Cstar = out["C_star"]
    converged = bool(out["converged"])
    err_hist = out.get("err_hist", None)

    # --- compute one-step objects at fixed point ---
    # 这里直接调用 solver.F(C*)，让它返回 Cnew 和 aux（包含 S, M）
    # 注意：如果 solver.F 的返回结构不同，你把这里对齐你当前实现即可
    Cnew, aux = solver.F(Cstar)
    S = aux.get("S", None)
    M = aux.get("M", None)

    # --- observables ---
    # residual memory q
    if HAVE_OLD_OBS:
        q = float(residual_memory(Cstar))
    else:
        q = residual_memory_fallback(Cstar)

    # focus sharpness kappa
    kappa = None
    M_sum = None
    if M is not None:
        kappa = float(focus_sharpness(M))
        M_sum = float(sanity_sum_rule(M))

    # decay classification + parameters
    rep = classify_phase(Cstar)
    # 可选：若你更信任自己现有的 corr_length_exp/power_law_alpha，也可以替换
    if HAVE_OLD_OBS:
        xi_est = float(corr_length_exp(Cstar))
        alpha_est = float(power_law_alpha(Cstar))
    else:
        xi_est = fit_exp_xi(Cstar, q=rep.q_hat)
        alpha_est = fit_power_alpha(Cstar, q=rep.q_hat)

    return dict(
        L=L,
        converged=converged,
        Cstar=Cstar,
        q=q,
        kappa=kappa,
        M_sum=M_sum,
        phase=rep.phase,
        fit_exp_r2=rep.fit_exp_r2,
        fit_pow_r2=rep.fit_pow_r2,
        xi=xi_est,
        alpha=alpha_est,
        err_hist=err_hist,
    )


# ============================================
# 3) Sweep L and collect results
# ============================================

results = []
for L in Ls:
    r = run_dmft_for_L(L)
    results.append(r)
    print(f"L={L:3d} | conv={r['converged']} | phase={r['phase']:<10} | q={r['q']:.3e} | kappa={r['kappa']}")

# Convert to arrays for plotting
Ls_arr = np.array([r["L"] for r in results], dtype=int)
conv_arr = np.array([r["converged"] for r in results], dtype=bool)

q_arr = np.array([r["q"] for r in results], dtype=float)
kappa_arr = np.array([np.nan if r["kappa"] is None else r["kappa"] for r in results], dtype=float)

xi_arr = np.array([np.nan if r["xi"] is None else r["xi"] for r in results], dtype=float)
alpha_arr = np.array([np.nan if r["alpha"] is None else r["alpha"] for r in results], dtype=float)

phase_list = [r["phase"] for r in results]


# ============================================
# 4) Plot: residual memory, decay indicator, focus sharpness
# ============================================

# 4.1 residual memory q(L)
plt.figure()
plt.plot(Ls_arr, q_arr, marker="o")
plt.xlabel("L")
plt.ylabel("residual memory  q")
plt.yscale("log")
plt.title(f"Residual memory vs L  (gamma={gamma}, init={init_name}, init_kwargs={init_kwargs})")
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.show()

# 4.2 decay: show both xi and alpha, plus a "chosen" indicator based on classify_phase
# We define a single y-value "decay_measure":
#   - if phase says exp/exp-ish: use xi
#   - if phase says powerlaw/powerlaw-ish: use alpha
#   - else NaN
decay_measure = np.full_like(q_arr, np.nan, dtype=float)
decay_label = []
for i, ph in enumerate(phase_list):
    if ph.startswith("exp"):
        decay_measure[i] = xi_arr[i]
        decay_label.append("xi")
    elif ph.startswith("powerlaw"):
        decay_measure[i] = alpha_arr[i]
        decay_label.append("alpha")
    elif ph == "plateau":
        decay_measure[i] = np.nan
        decay_label.append("plateau")
    else:
        decay_label.append("unclear")

plt.figure()
plt.plot(Ls_arr, xi_arr, marker="o")
plt.xlabel("L")
plt.ylabel("xi (exp fit)")
plt.title("Exponential correlation length estimate")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.show()

plt.figure()
plt.plot(Ls_arr, alpha_arr, marker="o")
plt.xlabel("L")
plt.ylabel("alpha (power-law fit)")
plt.title("Power-law exponent estimate")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.show()

# 4.3 focus sharpness kappa(L)
plt.figure()
plt.plot(Ls_arr, kappa_arr, marker="o")
plt.xlabel("L")
plt.ylabel("focus sharpness  kappa = (L+1) tr(M)")
plt.title("Focus sharpness vs L")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.show()

# Optional: a quick check that sum(M) ~ 1 (should be close to 1)
Msum_arr = np.array([np.nan if r["M_sum"] is None else r["M_sum"] for r in results], dtype=float)
plt.figure()
plt.plot(Ls_arr, Msum_arr, marker="o")
plt.xlabel("L")
plt.ylabel("sum(M)  (should be ~1)")
plt.title("Sanity check: sum(M)")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.show()

# Optional: show which phase label was assigned
# (simple text print; if you want, we can add a color-coded scatter later)
for L, ph in zip(Ls_arr, phase_list):
    print(f"L={L:3d}: phase={ph}")